In [3]:
import torch
import torch_directml

# Create DirectML device
dml = torch_directml.device()

print("PyTorch version:", torch.__version__)
print("Using device:", dml)

# Simple tensor test
x = torch.randn(1000, 1000, device=dml)
y = torch.randn(1000, 1000, device=dml)
z = torch.matmul(x, y)
print("Matrix multiply result shape:", z.shape)

PyTorch version: 2.4.1+cpu
Using device: privateuseone:0
Matrix multiply result shape: torch.Size([1000, 1000])


In [5]:
import torch
import torch_directml
import time

dml = torch_directml.device()
print("Device:", dml)

x = torch.randn(4096, 4096, device=dml)
y = torch.randn(4096, 4096, device=dml)

start = time.time()
for i in range(50):
    z = torch.matmul(x, y)
    if (i + 1) % 10 == 0:
        torch.directml.synchronize() if hasattr(torch, "directml") else None
        print(f"Step {i+1}")
end = time.time()

print("Total time:", end - start, "seconds")

Device: privateuseone:0
Step 10
Step 20
Step 30
Step 40
Step 50
Total time: 0.31407713890075684 seconds


In [11]:
import time
import torch
import torch_directml

# Choose device (DirectML = AMD GPU)
dml_device = torch_directml.device()
cpu_device = torch.device("cpu")


def log(s):
    print(f"[{time.strftime('%H:%M:%S')}] {s}", flush=True)


def matmul_stress(device, size=4096, iters=300):
    log(f"MatMul stress on {device}, size={size}x{size}, iters={iters}")

    # Allocate big tensors once
    x = torch.randn(size, size, device=device)
    y = torch.randn(size, size, device=device)

    # Warmup
    log("  Warmup...")
    for _ in range(10):
        z = x @ y

    # Timed loop
    log("  Timed loop...")
    start = time.time()
    for i in range(1, iters + 1):
        z = x @ y  # dependency chain keeps compute busy
        if i % 50 == 0:
            log(f"    Iter {i}/{iters}")
    end = time.time()

    total = end - start
    avg = total / iters
    log(f"  Finished matmul: total={total:.2f}s, per-iter={avg:.4f}s\n")


def cnn_stress(device, batch_size=32, iters=200):
    log(f"CNN stress on {device}, batch_size={batch_size}, iters={iters}")

    # Simple CNN model
    class SmallCNN(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = torch.nn.Conv2d(3, 64, 3, padding=1)
            self.conv2 = torch.nn.Conv2d(64, 128, 3, padding=1)
            self.conv3 = torch.nn.Conv2d(128, 256, 3, padding=1)
            self.pool = torch.nn.MaxPool2d(2)
            self.fc1 = torch.nn.Linear(256 * 28 * 28, 1024)
            self.fc2 = torch.nn.Linear(1024, 100)

        def forward(self, x):
            x = torch.relu(self.conv1(x))
            x = self.pool(x)
            x = torch.relu(self.conv2(x))
            x = self.pool(x)
            x = torch.relu(self.conv3(x))
            x = self.pool(x)  # 224 -> 112 -> 56 -> 28
            x = torch.flatten(x, 1)
            x = torch.relu(self.fc1(x))
            x = self.fc2(x)
            return x

    model = SmallCNN().to(device)
    optim = torch.optim.SGD(model.parameters(), lr=0.01)
    criterion = torch.nn.CrossEntropyLoss()

    # Synthetic data: big 224x224 images
    inputs = torch.randn(batch_size, 3, 224, 224, device=device)
    targets = torch.randint(0, 100, (batch_size,), device=device)

    # Warmup
    log("  Warmup...")
    for _ in range(5):
        optim.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optim.step()

    # Timed loop
    log("  Timed training loop...")
    start = time.time()
    for i in range(1, iters + 1):
        optim.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optim.step()

        if i % 20 == 0:
            log(f"    Iter {i}/{iters}, loss={loss.item():.4f}")
    end = time.time()

    total = end - start
    avg = total / iters
    log(f"  Finished CNN: total={total:.2f}s, per-iter={avg:.4f}s\n")


if __name__ == "__main__":
    log(f"PyTorch version: {torch.__version__}")
    log(f"DirectML device: {dml_device}")
    log(f"CPU device: {cpu_device}\n")

    # =======================
    # DirectML (AMD GPU) tests
    # =======================
    matmul_stress(dml_device, size=4096, iters=400)  # heavy matrix multiplies
    cnn_stress(dml_device, batch_size=32, iters=250)  # heavy CNN workload

    # If you also want to compare CPU vs GPU, uncomment below:
    # matmul_stress(cpu_device, size=2048, iters=100)
    # cnn_stress(cpu_device, batch_size=16, iters=80)

    log("All stress tests completed.")

ImportError: DLL load failed while importing torch_directml_native: The specified procedure could not be found.

In [10]:
import torch
import torch_directml

print(torch.__version__)
print(torch_directml.__version__)
print(torch.__file__)

ImportError: DLL load failed while importing torch_directml_native: The specified procedure could not be found.